# Agentes Locales: Razonamiento con Herramientas

## Objetivo

Crear un agente inteligente que pueda decidir cuándo usar herramientas externas (Tools) para responder preguntas complejas usando Llama 3 local.

### ¿Qué es un Agente?

Un **agente** es un sistema que puede:
1. **Pensar**: Analizar la pregunta del usuario
2. **Decidir**: Determinar si necesita usar herramientas
3. **Actuar**: Ejecutar herramientas cuando sea necesario
4. **Responder**: Generar una respuesta final basada en los resultados

### Flujo de un Agente:

```
Usuario: "¿Cuánto es 5 por 5?"
    ↓
Agente piensa: "Necesito multiplicar, tengo una herramienta para eso"
    ↓
Agente ejecuta: multiply(5, 5)
    ↓
Agente recibe: 25
    ↓
Agente responde: "5 por 5 es 25"
```

En este notebook construirás un agente completo con herramientas personalizadas.


In [ ]:
# Setup inicial
import sys
import os

# Hack para importar desde src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.models import get_local_llm

print("✓ Imports completados")


## Paso 1: Definición de Tools (Herramientas)

Las **tools** son funciones que el agente puede ejecutar para obtener información o realizar acciones. En LangChain, usamos el decorador `@tool` para crear herramientas.

### Importante: Los Docstrings

**Los docstrings son críticos**. El modelo LLM lee los docstrings para entender:
- Qué hace la herramienta
- Qué parámetros necesita
- Qué devuelve

Sin docstrings claros, el modelo no sabrá cómo usar la herramienta.


In [ ]:
from langchain_core.tools import tool

# Tool 1: Multiplicar dos números
@tool
def multiply(a: int, b: int) -> int:
    """Multiplica dos números enteros.
    
    Esta herramienta es útil cuando necesitas realizar multiplicaciones.
    
    Args:
        a: El primer número a multiplicar
        b: El segundo número a multiplicar
        
    Returns:
        El resultado de multiplicar a por b
        
    Example:
        multiply(5, 3) -> 15
    """
    return a * b

print("✓ Tool 'multiply' creada")
print(f"  Descripción: {multiply.description}")


In [ ]:
# Tool 2: Obtener el clima (simulado con datos falsos)
@tool
def get_current_weather(city: str) -> str:
    """Obtiene el clima actual de una ciudad.
    
    Esta herramienta devuelve información sobre la temperatura y condiciones
    climáticas de una ciudad específica.
    
    Args:
        city: El nombre de la ciudad (ej: "Madrid", "Barcelona", "Londres")
        
    Returns:
        Una cadena de texto con la temperatura en grados Celsius y condiciones
        
    Example:
        get_current_weather("Madrid") -> "La temperatura en Madrid es 22°C y está soleado"
    """
    # Datos simulados (en producción, esto llamaría a una API real)
    weather_data = {
        "Madrid": "La temperatura en Madrid es 22°C y está soleado",
        "Barcelona": "La temperatura en Barcelona es 20°C y está nublado",
        "Londres": "La temperatura en Londres es 15°C y está lloviendo",
        "París": "La temperatura en París es 18°C y está parcialmente nublado",
        "Berlín": "La temperatura en Berlín es 16°C y está soleado",
    }
    
    # Devolver datos simulados o un mensaje por defecto
    return weather_data.get(
        city, 
        f"La temperatura en {city} es 20°C y las condiciones son desconocidas"
    )

print("✓ Tool 'get_current_weather' creada")
print(f"  Descripción: {get_current_weather.description}")

# Probar la herramienta manualmente
print(f"\n✓ Prueba manual: {get_current_weather.invoke('Madrid')}")


### ¿Por qué los Docstrings son Importantes?

El modelo LLM (Llama 3) lee los docstrings para:

1. **Entender qué hace la herramienta**: El docstring describe la funcionalidad
2. **Saber qué parámetros necesita**: Los tipos y nombres de los argumentos
3. **Decidir cuándo usarla**: Compara la pregunta con la descripción
4. **Formatear la llamada correctamente**: Usa los nombres de parámetros del docstring

**Sin docstrings claros, el modelo no podrá usar las herramientas correctamente.**


## Paso 2: Binding (Vinculación) de Tools al Modelo

Para que el modelo pueda usar las herramientas, necesitamos **bindearlas** (vincularlas) al modelo. Esto le dice al modelo qué herramientas están disponibles.


In [ ]:
# Instanciar el modelo
llm = get_local_llm()

# Crear lista de herramientas
tools = [multiply, get_current_weather]

# Bindear las herramientas al modelo
llm_with_tools = llm.bind_tools(tools)

print("✓ Herramientas bindeadas al modelo")
print(f"  Herramientas disponibles: {len(tools)}")
for tool in tools:
    print(f"    - {tool.name}: {tool.description[:50]}...")


### Probar el Modelo con Tools

Ahora el modelo puede decidir cuándo usar herramientas. Veamos qué pasa cuando le preguntamos algo que requiere una herramienta:


In [ ]:
from langchain_core.messages import HumanMessage

# Pregunta que requiere usar la herramienta multiply
pregunta = "¿Cuánto es 5 por 5?"

print("=" * 60)
print("PRUEBA: Modelo con Tools")
print("=" * 60)
print(f"Pregunta: {pregunta}\n")

# Invocar el modelo
mensaje = HumanMessage(content=pregunta)
respuesta = llm_with_tools.invoke([mensaje])

print("Respuesta del modelo:")
print(f"  Tipo: {type(respuesta)}")
print(f"  Contenido: {respuesta.content}")
print(f"  Tool calls: {len(respuesta.tool_calls) if hasattr(respuesta, 'tool_calls') else 0}")

# Verificar si el modelo quiere usar una herramienta
if hasattr(respuesta, 'tool_calls') and respuesta.tool_calls:
    print("\n✓ El modelo decidió usar una herramienta:")
    for tool_call in respuesta.tool_calls:
        print(f"  - Tool: {tool_call['name']}")
        print(f"    Args: {tool_call['args']}")
else:
    print("\n⚠ El modelo respondió directamente sin usar herramientas")


### Entender las Tool Calls

Cuando el modelo decide usar una herramienta, en lugar de devolver texto, devuelve una **tool call** que contiene:
- `name`: Nombre de la herramienta a ejecutar
- `args`: Argumentos para la herramienta
- `id`: Identificador único de la llamada

El agente debe:
1. Detectar la tool call
2. Ejecutar la herramienta con los argumentos
3. Devolver el resultado al modelo
4. Permitir que el modelo genere la respuesta final


## Paso 3: Construcción del Agente

Un agente completo necesita:
1. **Modelo con tools**: Ya lo tenemos (`llm_with_tools`)
2. **Prompt del agente**: Instrucciones sobre cómo usar las herramientas
3. **AgentExecutor**: Gestiona el bucle de pensamiento-ejecución-respuesta

Usaremos `create_tool_calling_agent`, la forma moderna y recomendada en LangChain.


In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Crear el prompt del agente
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente útil que puede usar herramientas para responder preguntas.
    
Cuando necesites realizar cálculos, usa la herramienta 'multiply'.
Cuando necesites información del clima, usa la herramienta 'get_current_weather'.

Siempre explica tu razonamiento y los pasos que tomas."""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("✓ Prompt del agente creado")


### Componentes del Prompt

- **System message**: Instrucciones sobre cómo usar las herramientas
- **chat_history**: Historial de conversación (opcional)
- **human input**: La pregunta del usuario
- **agent_scratchpad**: Aquí LangChain inyecta automáticamente las tool calls y resultados


In [ ]:
# Crear el agente
agent = create_tool_calling_agent(
    llm=llm_with_tools,
    tools=tools,
    prompt=prompt
)

print("✓ Agente creado")
print(f"  Modelo: {type(llm_with_tools).__name__}")
print(f"  Tools: {len(tools)} herramientas disponibles")


### Crear el AgentExecutor

El `AgentExecutor` gestiona el bucle completo del agente:
1. Pasa el input al agente
2. Si el agente quiere usar una herramienta, la ejecuta
3. Devuelve el resultado al agente
4. Repite hasta que el agente genera una respuesta final
5. Devuelve la respuesta al usuario


In [ ]:
# Crear el AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # Mostrar el proceso de razonamiento
    handle_parsing_errors=True,  # Manejar errores de parsing
    max_iterations=5,  # Límite de iteraciones para evitar bucles infinitos
)

print("✓ AgentExecutor creado")
print("  verbose=True: Verás el proceso de razonamiento")
print("  max_iterations=5: Máximo 5 iteraciones por pregunta")


## Paso 4: Prueba de Razonamiento

Ahora probemos el agente con diferentes tipos de preguntas para ver cómo razona y usa las herramientas.


In [ ]:
# Prueba 1: Pregunta simple que requiere una herramienta
print("=" * 60)
print("PRUEBA 1: Multiplicación Simple")
print("=" * 60)

pregunta1 = "¿Cuánto es 5 por 5?"

resultado1 = agent_executor.invoke({"input": pregunta1})

print("\n" + "=" * 60)
print("RESPUESTA FINAL:")
print("=" * 60)
print(resultado1["output"])


In [ ]:
# Prueba 2: Pregunta sobre el clima
print("\n" + "=" * 60)
print("PRUEBA 2: Consulta del Clima")
print("=" * 60)

pregunta2 = "¿Qué tiempo hace en Madrid?"

resultado2 = agent_executor.invoke({"input": pregunta2})

print("\n" + "=" * 60)
print("RESPUESTA FINAL:")
print("=" * 60)
print(resultado2["output"])


In [ ]:
# Prueba 3: Pregunta compleja que requiere múltiples herramientas
print("=" * 60)
print("PRUEBA 3: Razonamiento Complejo")
print("=" * 60)
print("Pregunta que requiere:")
print("  1. Obtener el clima de Madrid")
print("  2. Extraer la temperatura")
print("  3. Multiplicarla por 2")
print("=" * 60)

pregunta3 = "¿Qué tiempo hace en Madrid y cuánto es esa temperatura multiplicada por 2?"

resultado3 = agent_executor.invoke({"input": pregunta3})

print("\n" + "=" * 60)
print("RESPUESTA FINAL:")
print("=" * 60)
print(resultado3["output"])


### Análisis del Proceso

Con `verbose=True`, deberías ver en la salida:

1. **Pensamiento inicial**: El agente analiza la pregunta
2. **Tool call 1**: Llama a `get_current_weather("Madrid")`
3. **Resultado 1**: Recibe "La temperatura en Madrid es 22°C..."
4. **Pensamiento**: Necesita extraer el número 22 y multiplicarlo por 2
5. **Tool call 2**: Llama a `multiply(22, 2)`
6. **Resultado 2**: Recibe 44
7. **Respuesta final**: Genera una respuesta combinando toda la información

Este es el poder de los agentes: **razonamiento multi-paso**.


In [ ]:
# Prueba 4: Pregunta que NO requiere herramientas
print("\n" + "=" * 60)
print("PRUEBA 4: Pregunta Sin Tools")
print("=" * 60)

pregunta4 = "¿Cómo estás?"

resultado4 = agent_executor.invoke({"input": pregunta4})

print("\n" + "=" * 60)
print("RESPUESTA FINAL:")
print("=" * 60)
print(resultado4["output"])
print("\n✓ El agente respondió directamente sin usar herramientas")


## Resumen: Componentes de un Agente

### Flujo Completo del Agente:

```
Usuario: "¿Cuánto es 5 por 5?"
    ↓
AgentExecutor recibe input
    ↓
Agente analiza: "Necesito multiplicar"
    ↓
Agente decide: Usar tool 'multiply'
    ↓
AgentExecutor ejecuta: multiply(5, 5) → 25
    ↓
Agente recibe resultado: 25
    ↓
Agente genera respuesta: "5 por 5 es 25"
    ↓
Usuario recibe respuesta final
```

### Componentes Clave:

1. ✅ **Tools**: Funciones que el agente puede ejecutar
2. ✅ **Docstrings**: Describen las tools para que el modelo las entienda
3. ✅ **bind_tools()**: Vincula las tools al modelo
4. ✅ **create_tool_calling_agent()**: Crea el agente con razonamiento
5. ✅ **AgentExecutor**: Gestiona el bucle de ejecución

### Ventajas de los Agentes:

- 🧠 **Razonamiento**: Pueden pensar antes de actuar
- 🔧 **Flexibilidad**: Deciden qué herramientas usar
- 🔄 **Multi-paso**: Pueden usar múltiples tools en secuencia
- 🎯 **Inteligencia**: Solo usan tools cuando es necesario


## Ejercicio Práctico

### Tu Turno

1. **Crea una nueva herramienta**:
   - Por ejemplo: `calculate_area(length: int, width: int)` para calcular el área de un rectángulo
   - Asegúrate de tener un docstring claro

2. **Añádela al agente**:
   - Agrega la nueva tool a la lista `tools`
   - Recrea el agente y el executor

3. **Prueba preguntas complejas**:
   - "Calcula el área de un rectángulo de 5 por 3 y luego multiplícalo por 2"
   - "¿Cuál es el área de un cuadrado de lado 10?"

### Pistas:

- Usa el decorador `@tool` para crear tu herramienta
- Incluye un docstring descriptivo
- Prueba con diferentes combinaciones de tools


In [ ]:
# Tu código aquí
# Crea tu propia herramienta y prueba el agente

# @tool
# def calculate_area(length: int, width: int) -> int:
#     """Calcula el área de un rectángulo.
#     
#     Args:
#         length: La longitud del rectángulo
#         width: El ancho del rectángulo
#         
#     Returns:
#         El área del rectángulo (length * width)
#     """
#     return length * width

# # Agregar a la lista de tools y recrear el agente
# # tools = [multiply, get_current_weather, calculate_area]
# # ...


## Notas Técnicas

### Tipos de Tools

Puedes crear tools de diferentes formas:

1. **Con decorador `@tool`**: La forma más simple
2. **Con `StructuredTool`**: Para más control
3. **Tools de LangChain**: Herramientas pre-construidas (búsqueda web, calculadora, etc.)

### Manejo de Errores

El `AgentExecutor` tiene protección contra:
- **Bucle infinito**: `max_iterations` limita las iteraciones
- **Errores de parsing**: `handle_parsing_errors=True` maneja errores
- **Tool failures**: Puedes manejar errores en las tools

### Combinar con RAG y Memoria

Puedes combinar agentes con:
- **RAG**: Para que el agente tenga acceso a documentos
- **Memoria**: Para conversaciones con contexto
- **Múltiples tools**: Para capacidades más complejas

### Optimizaciones

- **Caching**: Cachear resultados de tools costosas
- **Parallel execution**: Ejecutar tools en paralelo cuando sea posible
- **Streaming**: Usar streaming para respuestas en tiempo real


## Troubleshooting

### El modelo no usa las herramientas

- ✅ Verifica que los docstrings sean claros y descriptivos
- ✅ Asegúrate de que `bind_tools()` se haya ejecutado correctamente
- ✅ Revisa que el prompt incluya instrucciones sobre cuándo usar tools

### El agente entra en bucle

- ✅ Reduce `max_iterations` para limitar las iteraciones
- ✅ Mejora el prompt para dar instrucciones más claras
- ✅ Verifica que las tools devuelvan resultados válidos

### Errores de tipo en las tools

- ✅ Asegúrate de que los tipos de parámetros sean correctos
- ✅ Usa type hints en las funciones de las tools
- ✅ Valida los argumentos antes de ejecutar
